# GRPO Training for Methanol APC Environment

**Goal**: Train an LLM to act as an autonomous Advanced Process Control (APC) operator for a methanol synthesis reactor using Group Relative Policy Optimization (GRPO).

**Architecture**: `LLM generates JSON action` â†’ `Environment physics engine` â†’ `Dense reward (yield + safety + profit)` â†’ `GRPO policy update`

**Environment**: [HF Space](https://huggingface.co/spaces/glitchfilter/methanol-apc-env) | [GitHub](https://github.com/Bhavneet1492/openenv-methanol-apc)

**Key Innovation**: The reward comes from a real chemical engineering simulation â€” not a heuristic. The agent must learn thermodynamics.

---
## 1. Setup and Dependencies

In [ ]:
%%capture
# Install dependencies (Colab / HF Space)
import subprocess, sys, os, pathlib

# Core ML
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl>=0.15" peft accelerate bitsandbytes
!pip install -q "openenv-core[core]>=0.2.2" numpy matplotlib datasets

# Clone environment repo
_repo = pathlib.Path('/content/methanol-apc')
if not _repo.exists():
    os.system('git clone https://github.com/Bhavneet1492/openenv-methanol-apc.git /content/methanol-apc')
sys.path.insert(0, str(_repo))
sys.path.insert(0, str(_repo / 'methanol_apc_env' / 'server'))
sys.path.insert(0, str(_repo / 'methanol_apc_env'))
print('Setup complete')

In [ ]:
import json, os, random, time
import numpy as np
import matplotlib.pyplot as plt
import torch

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {vram_gb:.1f} GB")
else:
    vram_gb = 0
    print("WARNING: No GPU. Training will be very slow.")

## 2. Load Model with Unsloth

Model loaded FIRST because tokenizer is needed to format prompts.

Auto-selects: Qwen2.5-3B for T4 (16GB), Qwen2.5-7B for A100 (40GB+).

In [ ]:
from unsloth import FastLanguageModel

# Auto-select model based on VRAM
if vram_gb >= 30:
    MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
elif vram_gb >= 10:
    MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
else:
    MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"
print(f"Selected: {MODEL_NAME} (VRAM: {vram_gb:.0f}GB)")

MAX_SEQ_LENGTH = 2048
LORA_R, LORA_ALPHA = 16, 32

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH,
    dtype=None, load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=LORA_R, lora_alpha=LORA_ALPHA,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_dropout=0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=42,
)
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 3. Environment, Reward Function, and Prompts

The reward function runs the **actual physics simulation** â€” LHHW kinetics, SRK EOS, RK4 ODE solver. Each completion is parsed as JSON, stepped through the reactor, and scored on yield + safety + profit.

In [ ]:
from methanol_apc_env.server.methanol_environment import MethanolAPCEnvironment
from methanol_apc_env.models import MethanolAPCAction

# === Config ===
NUM_TRAIN_STEPS = 150
GROUP_SIZE = 4
BATCH_SIZE = 2
GRAD_ACCUM = 4
LEARNING_RATE = 5e-6
MAX_COMPLETION_LENGTH = 128
TEMPERATURE = 0.7
TASKS = ["optimization", "startup", "disturbance_rejection"]
NUM_PROMPTS = 200
PLOT_DIR = "./training_plots"
os.makedirs(PLOT_DIR, exist_ok=True)

SYSTEM_PROMPT = """You are an AI controller for a methanol synthesis reactor (ICI Low-Pressure Process).
Given sensor readings, output a JSON control action:
{"feed_rate_h2": <0-10>, "feed_rate_co": <0-5>, "cooling_water_flow": <0-100>, "compressor_power": <0-100>}

RULES:
- CO + 2H2 -> CH3OH is exothermic. More feed = more heat + methanol.
- Optimal: 240-260C. >270C = catalyst damage. >300C = SHUTDOWN.
- H2/CO ratio ~ 2.0. Cooling removes heat. Higher pressure = faster reaction.
- Revenue: $0.74/kg methanol. Costs: feed + electricity + cooling.

Output ONLY the JSON object."""

def make_env(task="optimization", seed=42):
    env = MethanolAPCEnvironment()
    obs = env.reset(task_name=task, seed=seed)
    return env, obs

def obs_to_text(obs):
    return (
        f"T={obs.temperature:.1f}C P={obs.pressure:.1f}bar "
        f"H2={obs.feed_rate_h2:.2f} CO={obs.feed_rate_co:.2f} ratio={obs.h2_co_ratio:.2f} "
        f"cool={obs.cooling_water_flow:.0f}L/min cat={obs.catalyst_health:.2%} "
        f"rate={obs.reaction_rate:.4f} MeOH={obs.methanol_produced:.1f}kg "
        f"profit=${obs.cumulative_profit:.2f} step={obs.step_number}/{obs.max_steps}"
    )

def _replay_warmup(env, seed, num_warmup):
    for step in range(num_warmup):
        rng = random.Random(seed * 1000 + step)
        action = MethanolAPCAction(
            feed_rate_h2=rng.uniform(1, 8), feed_rate_co=rng.uniform(0.5, 4),
            cooling_water_flow=rng.uniform(10, 80), compressor_power=rng.uniform(30, 80),
        )
        obs = env.step(action)
        if obs.done: break

env, obs = make_env("optimization")
print(f"Sample observation:\n{obs_to_text(obs)}")

In [ ]:
def reward_fn(completions, task=None, seed=None, num_warmup=None, **kwargs):
    """Score completions by stepping the physics sim. Returns list[float]."""
    rewards = []
    for i, completion in enumerate(completions):
        t  = task[i] if task is not None else random.choice(TASKS)
        s  = int(seed[i]) if seed is not None else 42
        nw = int(num_warmup[i]) if num_warmup is not None else 0
        try:
            text = completion if isinstance(completion, str) else str(completion)
            text = text.strip()
            if '```' in text:
                text = text.split('```')[1].replace('json','',1).strip()
            start, end = text.find('{'), text.rfind('}') + 1
            if start >= 0 and end > start:
                text = text[start:end]
            action = MethanolAPCAction(**json.loads(text))
            env, _ = make_env(task=t, seed=s)
            if nw > 0: _replay_warmup(env, s, nw)
            obs = env.step(action)
            r = max(0.01, min(0.99, float(obs.reward)))
            rewards.append(r * 0.9 + 0.1)  # Valid JSON bonus
        except Exception:
            rewards.append(0.01)
    return rewards

# Test
print(f"Good action:  {reward_fn(['{"feed_rate_h2":5,"feed_rate_co":2.5,"cooling_water_flow":50,"compressor_power":60}'], task=['optimization'], seed=[42], num_warmup=[0])[0]:.4f}")
print(f"Bad (no JSON): {reward_fn(['garbage'], task=['optimization'], seed=[42], num_warmup=[0])[0]:.4f}")

In [ ]:
from datasets import Dataset

def build_prompt_dataset(num_prompts=NUM_PROMPTS):
    prompts = []
    for i in range(num_prompts):
        task = TASKS[i % len(TASKS)]
        seed = i
        nw = random.randint(0, 5)
        env, obs = make_env(task=task, seed=seed)
        actual = 0
        for step in range(nw):
            rng = random.Random(seed * 1000 + step)
            obs = env.step(MethanolAPCAction(
                feed_rate_h2=rng.uniform(1,8), feed_rate_co=rng.uniform(0.5,4),
                cooling_water_flow=rng.uniform(10,80), compressor_power=rng.uniform(30,80)))
            actual += 1
            if obs.done: break
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Sensors:\n{obs_to_text(obs)}\n\nAction JSON:"},
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        prompts.append({"prompt": prompt, "task": task, "seed": seed, "num_warmup": actual})
    return Dataset.from_list(prompts)

print("Building prompts...")
dataset = build_prompt_dataset(NUM_PROMPTS)
print(f"Dataset: {len(dataset)} prompts, tasks: {dict(zip(*np.unique(dataset['task'], return_counts=True)))}")

## 4. GRPO Training

Each step: sample prompt â†’ generate GROUP_SIZE actions â†’ score with physics â†’ GRPO update.

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from transformers import TrainerCallback

class RewardLogger(TrainerCallback):
    def __init__(self): self.rewards, self.losses = [], []
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs: return
        s = state.global_step
        loss = logs.get('loss')
        rew = logs.get('reward', logs.get('rewards/mean', logs.get('reward/mean')))
        p = [f'[Step {s:>4d}]']
        if loss is not None: p.append(f'loss={loss:.4f}'); self.losses.append({'step':s,'loss':loss})
        if rew is not None: p.append(f'reward={rew:.4f}'); self.rewards.append({'step':s,'reward':rew})
        print('  '+' '.join(p))

logger = RewardLogger()

args = GRPOConfig(
    output_dir='./grpo_output', max_steps=NUM_TRAIN_STEPS,
    per_device_train_batch_size=BATCH_SIZE, gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE, max_completion_length=MAX_COMPLETION_LENGTH,
    num_generations=GROUP_SIZE, temperature=TEMPERATURE,
    logging_steps=5, save_steps=50, report_to='none',
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(), seed=42,
)

trainer = GRPOTrainer(
    model=model, args=args, train_dataset=dataset,
    reward_funcs=reward_fn, processing_class=tokenizer, callbacks=[logger],
)

print(f"Training: {MODEL_NAME}, {NUM_TRAIN_STEPS} steps, batch={BATCH_SIZE*GRAD_ACCUM}")
t0 = time.time()
result = trainer.train()
print(f"\nDone in {(time.time()-t0)/60:.1f}min. Loss: {result.training_loss:.4f}")

## 5. Training Curves and Evaluation

In [ ]:
# Loss curve
h = trainer.state.log_history
steps = [e['step'] for e in h if 'loss' in e]
losses = [e['loss'] for e in h if 'loss' in e]

fig, ax = plt.subplots(figsize=(10,5))
ax.plot(steps, losses, '#3b82f6', lw=2, label='Loss')
if len(steps)>10:
    w = max(3, len(losses)//10)
    sm = np.convolve(losses, np.ones(w)/w, 'valid')
    ax.plot(steps[w-1:], sm, '#1e40af', lw=2, ls='--', label=f'Smoothed')
ax.set(xlabel='Step', ylabel='Loss', title=f'GRPO Loss â€” {MODEL_NAME.split("/")[-1]}')
ax.legend(); ax.grid(alpha=0.3); fig.tight_layout()
fig.savefig(f'{PLOT_DIR}/loss_curve.png', dpi=150); plt.show()

In [ ]:
# Evaluate baseline vs trained
def eval_agent(model, tok, task='optimization', eps=5, steps=15):
    FastLanguageModel.for_inference(model)
    all_r = []
    for ep in range(eps):
        env, obs = make_env(task, ep*100)
        rs = []
        for _ in range(steps):
            if obs.done: break
            msgs = [{'role':'system','content':SYSTEM_PROMPT},
                    {'role':'user','content':f'Sensors:\n{obs_to_text(obs)}\n\nAction JSON:'}]
            p = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
            inp = tok(p, return_tensors='pt').to(model.device)
            with torch.no_grad():
                out = model.generate(**inp, max_new_tokens=150, temperature=0.3,
                                     do_sample=True, pad_token_id=tok.eos_token_id)
            resp = tok.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True)
            try:
                t = resp.strip(); s,e = t.find('{'), t.rfind('}')+1
                obs = env.step(MethanolAPCAction(**json.loads(t[s:e])))
                rs.append(float(obs.reward))
            except: obs = env.step(MethanolAPCAction(feed_rate_h2=3,feed_rate_co=1.5,cooling_water_flow=60,compressor_power=50)); rs.append(float(obs.reward))
        all_r.append(rs)
    ml = max(len(r) for r in all_r)
    return np.mean([r+[r[-1]]*(ml-len(r)) for r in all_r], axis=0)

def eval_baseline(task='optimization', eps=5, steps=15):
    all_r = []
    for ep in range(eps):
        env, obs = make_env(task, ep*100); rs = []
        for _ in range(steps):
            if obs.done: break
            obs = env.step(MethanolAPCAction(feed_rate_h2=random.uniform(1,8),feed_rate_co=random.uniform(0.5,4),
                cooling_water_flow=random.uniform(10,80),compressor_power=random.uniform(20,80)))
            rs.append(float(obs.reward))
        all_r.append(rs)
    ml = max(len(r) for r in all_r)
    return np.mean([r+[r[-1]]*(ml-len(r)) for r in all_r], axis=0)

print('Evaluating...')
bl = eval_baseline(); tr = eval_agent(model, tokenizer)
print(f'Baseline: {np.mean(bl):.4f}, Trained: {np.mean(tr):.4f}, Delta: {np.mean(tr)-np.mean(bl):+.4f}')

In [ ]:
# Reward + comparison plots
fig, ax = plt.subplots(figsize=(10,5))
ax.plot(range(len(tr)), tr, '#10b981', lw=2, label=f'Trained (avg:{np.mean(tr):.3f})')
ax.axhline(np.mean(tr), color='#10b981', ls='--', alpha=0.5)
ax.set(xlabel='Step', ylabel='Reward', title='GRPO Trained Agent â€” Reward per Step')
ax.legend(); ax.grid(alpha=0.3); fig.tight_layout()
fig.savefig(f'{PLOT_DIR}/reward_curve.png', dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(10,5))
ax.plot(range(len(bl)), bl, '#ef4444', lw=2, alpha=0.8, label=f'Random ({np.mean(bl):.3f})')
ax.plot(range(len(tr)), tr, '#10b981', lw=2, label=f'GRPO Trained ({np.mean(tr):.3f})')
ax.fill_between(range(len(bl)), bl, alpha=0.1, color='#ef4444')
ax.fill_between(range(len(tr)), tr, alpha=0.1, color='#10b981')
ax.set(xlabel='Step', ylabel='Reward', title='Baseline vs GRPO-Trained â€” Methanol APC')
ax.legend(loc='lower right'); ax.grid(alpha=0.3); fig.tight_layout()
fig.savefig(f'{PLOT_DIR}/baseline_vs_trained.png', dpi=150); plt.show()

imp = np.mean(tr)-np.mean(bl)
print(f'\nIMPROVEMENT: {imp:+.4f} ({imp/max(np.mean(bl),1e-6)*100:+.1f}%)')

In [ ]:
# Save model
model.save_pretrained('./grpo_methanol_trained')
tokenizer.save_pretrained('./grpo_methanol_trained')
for f in ['loss_curve.png','reward_curve.png','baseline_vs_trained.png']:
    p = f'{PLOT_DIR}/{f}'
    if os.path.exists(p): print(f'  {f}: {os.path.getsize(p)/1024:.0f}KB')
print(f'\nDone! Baseline={np.mean(bl):.4f} Trained={np.mean(tr):.4f} Delta={imp:+.4f}')